In [ ]:
!pip freeze | grep scikit-learn

In [ ]:
!python -V

In [ ]:
import pickle
import pandas as pd
import os

In [ ]:
with open('/workspaces/MLOps/01-intro/models/lin_reg.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [ ]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [ ]:
df = read_data('yellow_tripdata_2023-03.parquet')
print(df.head())

year = df["tpep_pickup_datetime"].dt.year.values[0]
month = df["tpep_pickup_datetime"].dt.month.values[0]
print(f"year={year}, month={month}")

In [ ]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

question-1

In [ ]:
df.duration.std()

Question-2

In [ ]:
year = 2023
month = 3
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')


In [ ]:
df_results = pd.DataFrame({
    'ride_id': df.ride_id,
    'predicted_duration': y_pred
})

print(df_results.head())

output_file = f'yellow_tripdata_{year:04d}-{month:02d}_predictions.parquet'
df_results.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)


In [ ]:
size = os.path.getsize("yellow_tripdata_2023-03_predictions.parquet")
print("The size of the predictions dataframe file is: ",size/ (1024 * 1024), "MB", f"and {size} bytes")

Testing

In [4]:
import pandas as pd

In [ ]:
input = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet'
df = pd.read_parquet(input)
print(df.head())